In [ ]:
# Databricks notebook source

In [ ]:
print("Checking secrets...")
errors = []

secrets_to_check = [
    "alpha-vantage-api-key",
    "sp-client-id",
    "sp-tenant-id",
    "sp-client-secret"
]

for secret in secrets_to_check:
    try:
        val = dbutils.secrets.get(scope="retail-banking-scope", key=secret)
        if val and len(val) > 0:
            print(f"  ✅ {secret} — found ({len(val)} chars)")
        else:
            print(f"  ❌ {secret} — empty value!")
            errors.append(secret)
    except Exception as e:
        print(f"  ❌ {secret} — ERROR: {str(e)}")
        errors.append(secret)

if errors:
    raise Exception(f"Fix these secrets first: {errors}")
else:
    print("\n✅ CHECK 1 PASSED — All secrets readable from Key Vault")

In [ ]:
print("Checking ADLS Gen2 connection...")

SP_CLIENT_ID     = dbutils.secrets.get(scope="retail-banking-scope", key="sp-client-id")
SP_TENANT_ID     = dbutils.secrets.get(scope="retail-banking-scope", key="sp-tenant-id")
SP_CLIENT_SECRET = dbutils.secrets.get(scope="retail-banking-scope", key="sp-client-secret")
STORAGE_ACCOUNT  = "retailbankingdl"

try:
    spark.conf.set(
        f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
        "OAuth"
    )
    spark.conf.set(
        f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net",
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
    )
    spark.conf.set(
        f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net",
        SP_CLIENT_ID
    )
    spark.conf.set(
        f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net",
        SP_CLIENT_SECRET
    )
    spark.conf.set(
        f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net",
        f"https://login.microsoftonline.com/{SP_TENANT_ID}/oauth2/token"
    )
    print("  ✅ Spark config set successfully")
except Exception as e:
    raise Exception(f"❌ Spark config failed: {str(e)}")

In [ ]:
print("Checking ADLS Bronze container access...")

ADLS_BRONZE_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net"

try:
    files = dbutils.fs.ls(ADLS_BRONZE_PATH)
    print(f"  ✅ Bronze container accessible — {len(files)} item(s) found")
    for f in files:
        print(f"     📁 {f.name}")
except Exception as e:
    error_msg = str(e)
    if "BlobNotFound" in error_msg or "ResourceNotFound" in error_msg:
        print("  ✅ Bronze container accessible — empty (expected on first run)")
    elif "AuthorizationPermissionMismatch" in error_msg:
        raise Exception(
            "❌ Service Principal does not have access to ADLS.\n"
            "Fix: Go to Storage Account → IAM → Add role assignment → "
            "'Storage Blob Data Contributor' → assign to 'retail-banking-sp'"
        )
    elif "InvalidClientSecretCredential" in error_msg or "AADSTS" in error_msg:
        raise Exception(
            "❌ Service Principal credentials are wrong.\n"
            "Fix: Check sp-client-id, sp-tenant-id, sp-client-secret in Key Vault"
        )
    else:
        raise Exception(f"❌ ADLS access failed: {error_msg}")

print("\n✅ CHECK 3 PASSED — ADLS Gen2 Bronze container is accessible")

In [ ]:
print("Checking write access to ADLS Bronze...")

test_path = f"{ADLS_BRONZE_PATH}/_validation/test"

try:
    test_df = spark.createDataFrame([("validation_test", "ok")], ["key", "value"])
    test_df.write.format("delta").mode("overwrite").save(test_path)
    print("  ✅ Write successful")

    # Read it back
    read_df = spark.read.format("delta").load(test_path)
    count = read_df.count()
    print(f"  ✅ Read back successful — {count} row(s)")

    # Clean up
    dbutils.fs.rm(test_path, recurse=True)
    print("  ✅ Test data cleaned up")

except Exception as e:
    raise Exception(f"❌ Write test failed: {str(e)}")

print("\n✅ CHECK 4 PASSED — Write and read to ADLS confirmed")

In [ ]:
print("Checking Alpha Vantage API...")

import requests

ALPHA_VANTAGE_API_KEY = dbutils.secrets.get(scope="retail-banking-scope", key="alpha-vantage-api-key")

try:
    # Use a lightweight endpoint to validate the key
    response = requests.get(
        "https://www.alphavantage.co/query",
        params={
            "function": "CURRENCY_EXCHANGE_RATE",
            "from_currency": "EUR",
            "to_currency": "USD",
            "apikey": ALPHA_VANTAGE_API_KEY
        },
        timeout=15
    )
    response.raise_for_status()
    data = response.json()

    if "Error Message" in data:
        raise Exception(f"❌ Invalid API key: {data['Error Message']}")
    elif "Note" in data:
        raise Exception(f"❌ Rate limit hit: {data['Note']}")
    elif "Information" in data:
        raise Exception(f"❌ API limit: {data['Information']}")
    elif "Realtime Currency Exchange Rate" in data:
        rate = data["Realtime Currency Exchange Rate"]["5. Exchange Rate"]
        print(f"  ✅ API key valid — EUR/USD rate: {rate}")
    else:
        print(f"  ✅ API responded — keys: {list(data.keys())}")

except requests.exceptions.Timeout:
    raise Exception("❌ Alpha Vantage API timed out — check internet connectivity")
except requests.exceptions.ConnectionError:
    raise Exception("❌ Cannot reach Alpha Vantage — check network/firewall")

print("\n✅ CHECK 5 PASSED — Alpha Vantage API key is valid and reachable")

In [ ]:
print("=" * 50)
print("PRE-FLIGHT VALIDATION COMPLETE")
print("=" * 50)
print("  ✅ Check 1 — All secrets readable from Key Vault")
print("  ✅ Check 2 — ADLS Gen2 Spark config set")
print("  ✅ Check 3 — Bronze container accessible")
print("  ✅ Check 4 — Write/read to ADLS confirmed")
print("  ✅ Check 5 — Alpha Vantage API key valid")
print("=" * 50)
print("You are ready to run 01_bronze_alpha_vantage notebook.")
print("=" * 50)